# 01 — Autograd from Scratch

The goal of this lesson is to understand reverse-mode automatic
differentiation by implementing a minimal scalar autograd engine.

We start with scalar values before moving back to tensors.

In [24]:
class Value:
    def __init__(self, data):
        self.data = data
        self.grad = 0.0


a = Value(2.0)

print(a.data)
print(a.grad)

2.0
0.0


In [25]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0

        self._prev = set(_children)  # graph connectivity
        self._op = _op

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")
        return out

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


a = Value(2.0)
b = Value(3.0)

c = a + b

print(c.data)
print(c._op)
print(c._prev)

5.0
+
{Value(data=2.0, grad=0.0), Value(data=3.0, grad=0.0)}


In [26]:
# add `backward`
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0

        self._prev = set(_children)  # graph connectivity
        self._op = _op
        self._backward = lambda: None

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad  # grad addition
            other.grad += out.grad  # add op `grad` = 1

        out._backward = _backward

        return out

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


# test 1
a = Value(2.0)
b = Value(3.0)

c = a + b
c.grad = 1.0
c._backward()
print("a:", a)
print("b:", b)
print("c:", c)

# test 2
a = Value(2.0)
c = a + a
c.grad = 1.0
c._backward()

print(a)
print(c)

a: Value(data=2.0, grad=1.0)
b: Value(data=3.0, grad=1.0)
c: Value(data=5.0, grad=1.0)
Value(data=2.0, grad=2.0)
Value(data=4.0, grad=1.0)


## Addition and Multiplication: Local Gradients

Each operation in the computation graph knows how to propagate an
incoming gradient to its direct inputs.

### Addition

For

$$
c = a + b,
$$

the local derivatives are

$$
\frac{\partial c}{\partial a} = 1,
\qquad
\frac{\partial c}{\partial b} = 1.
$$

If the upstream gradient is

$$
\frac{\partial L}{\partial c},
$$

then by the chain rule,

$$
\frac{\partial L}{\partial a}
=
\frac{\partial L}{\partial c}
\frac{\partial c}{\partial a}
=
\frac{\partial L}{\partial c}.
$$

Similarly,

$$
\frac{\partial L}{\partial b}
=
\frac{\partial L}{\partial c}.
$$

### Multiplication

For

$$
c = ab,
$$

the local derivatives are

$$
\frac{\partial c}{\partial a} = b,
\qquad
\frac{\partial c}{\partial b} = a.
$$

Therefore,

$$
\frac{\partial L}{\partial a}
=
\frac{\partial L}{\partial c} b,
$$

and

$$
\frac{\partial L}{\partial b}
=
\frac{\partial L}{\partial c} a.
$$

The key idea is that each operation only needs to know its
**local derivative**. The complete gradient is obtained by combining
local derivatives through the chain rule.

In [27]:
# multificaiton
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0

        self._prev = set(_children)  # graph connectivity
        self._op = _op
        self._backward = lambda: None

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad  # grad addition
            other.grad += out.grad  # add op `grad` = 1

        out._backward = _backward

        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


a = Value(2.0)
b = Value(3.0)

c = a * b

c.grad = 1.0
c._backward()

print("a:", a)
print("b:", b)
print("c:", c)

a: Value(data=2.0, grad=3.0)
b: Value(data=3.0, grad=2.0)
c: Value(data=6.0, grad=1.0)


## Backpropagation Through the Computation Graph

Calling `_backward()` on a single node only propagates the gradient
through one local operation.

Consider the computation

$$
d = a + b
$$

followed by

$$
L = dc.
$$

The computation graph is:

<pre>
a ──┐
    + → d ──┐
b ──┘       × → L
        c ──┘
</pre>

The complete expression is

$$
L = (a+b)c.
$$

The gradients are

$$
\frac{\partial L}{\partial a} = c,
$$

$$
\frac{\partial L}{\partial b} = c,
$$

and

$$
\frac{\partial L}{\partial c} = a+b.
$$

Calling `_backward()` only on `L` propagates the gradient through the
multiplication node, but it does not automatically continue through
the addition node.

Reverse-mode automatic differentiation must therefore traverse the
entire computation graph from the final output back toward the leaf
nodes.

### Why do we need a topological order?

A node should execute its local `_backward()` function only after it
has received all gradient contributions from nodes that depend on it.

For example, a valid forward topological order could be:

<pre>
a → b → d → c → L
</pre>

Backpropagation then processes this order in reverse:

<pre>
L → c → d → b → a
</pre>

This ensures that gradients are accumulated before they are propagated
further backward.

In [28]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0

        self._prev = set(_children)  # graph connectivity
        self._op = _op
        self._backward = lambda: None

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad  # grad addition
            other.grad += out.grad  # add op `grad` = 1

        out._backward = _backward

        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)

                for parent in node._prev:
                    build_topo(parent)

                topo.append(node)

        build_topo(self)

        self.grad = 1.0

        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


# test
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

d = a + b
L = d * c

L.backward()

print("a:", a)
print("b:", b)
print("c:", c)
print("d:", d)
print("L:", L)

a: Value(data=2.0, grad=4.0)
b: Value(data=3.0, grad=4.0)
c: Value(data=4.0, grad=5.0)
d: Value(data=5.0, grad=4.0)
L: Value(data=20.0, grad=1.0)


## Local Backward Rules vs Global Backpropagation

The computation graph is created during the **forward pass**.

For example,

$$
d = a + b
$$

creates a node `d` that stores references to `a` and `b`, while

$$
L = dc
$$

creates another node that depends on `d` and `c`.

Therefore, by the time `L.backward()` is called, the computation graph
already exists.

The `backward()` method does not create the graph. Instead, it:

1. discovers the nodes reachable from the output,
2. builds a topological ordering,
3. initializes the output gradient,
4. traverses the graph in reverse order,
5. calls each node's local `_backward()` function.

The two backward mechanisms have different responsibilities.

### `_backward()`: local derivative rule

Each operation knows how to propagate an incoming gradient to its
direct inputs.

For multiplication,

$$
z = xy,
$$

the local derivatives are

$$
\frac{\partial z}{\partial x}=y,
\qquad
\frac{\partial z}{\partial y}=x.
$$

Therefore,

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial z}
\frac{\partial z}{\partial x}
=
\frac{\partial L}{\partial z}y.
$$

The `_backward()` function implements this local rule.

### `backward()`: global graph traversal

The global `backward()` method does not need to know the derivative of
addition, multiplication, or any other operation.

Its job is to execute local backward rules in the correct order.

A useful distinction is:

<pre>
_backward()  → HOW does this operation propagate gradients?
backward()   → WHEN should each operation propagate gradients?
</pre>

Automatic differentiation works because complex expressions are built
from simpler primitive operations whose local derivatives are known.
The chain rule composes these local derivative rules across the entire
computation graph.

## Basic Scalar Operations

A small autograd engine only needs derivative rules for a set of
primitive scalar operations.

More complex expressions can then be constructed by composing these
operations.

For each operation, the forward pass computes a value and stores a
local `_backward()` function.

### Addition

$$
z = x + y
$$

with

$$
\frac{\partial z}{\partial x} = 1,
\qquad
\frac{\partial z}{\partial y} = 1.
$$

### Multiplication

$$
z = xy
$$

with

$$
\frac{\partial z}{\partial x} = y,
\qquad
\frac{\partial z}{\partial y} = x.
$$

### Power

For a constant exponent $n$,

$$
z = x^n,
$$

the derivative is

$$
\frac{\partial z}{\partial x}
=
n x^{n-1}.
$$

### Exponential

$$
z = e^x
$$

has the derivative

$$
\frac{\partial z}{\partial x}
=
e^x
=
z.
$$

### Logarithm

$$
z = \log x
$$

has the derivative

$$
\frac{\partial z}{\partial x}
=
\frac{1}{x}.
$$

### Hyperbolic tangent

$$
z = \tanh(x)
$$

has the derivative

$$
\frac{\partial z}{\partial x}
=
1-z^2.
$$

### ReLU

$$
z = \max(0,x).
$$

For $x>0$,

$$
\frac{\partial z}{\partial x}=1,
$$

and for $x<0$,

$$
\frac{\partial z}{\partial x}=0.
$$

ReLU is not differentiable exactly at $x=0$.
A practical implementation must choose a convention; here we use a
gradient of $0$ at that point.

Subtraction and division do not require new derivative rules. They can
be expressed using negation, multiplication, and powers:

$$
x-y = x + (-y)
$$

and

$$
\frac{x}{y} = x y^{-1}.
$$

This illustrates an important idea: an autograd system only needs
derivative rules for primitive operations. Composite operations can
reuse them through the computation graph.

In [29]:
import math


class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0

        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        out = Value(
            self.data + other.data,
            (self, other),
            "+",
        )

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        out = Value(
            self.data * other.data,
            (self, other),
            "*",
        )

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float))

        out = Value(self.data**exponent, (self,), f"**{exponent}")

        def _backward():
            self.grad += exponent * self.data ** (exponent - 1) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward

        return out

    def log(self):
        out = Value(math.log(self.data), (self,), "log")

        def _backward():
            self.grad += (1 / self.data) * out.grad

        out._backward = _backward

        return out

    def tanh(self) -> "Value":
        t: float = math.tanh(self.data)

        out: Value = Value(
            t,
            (self,),
            "tanh",
        )

        def _backward() -> None:
            self.grad += (1.0 - t**2) * out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "ReLU")

        def _backward():
            self.grad += 1.0 if self.data > 0 else 0.0 * out.grad

        out._backward = _backward
        return out

    # neg value
    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    # x/y = x(y**-1)
    def __truediv__(self, other):
        return self * (other**-1)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return other + (-self)

    def __rtruediv__(self, other):
        return other * (self**-1)

    def backward(self):
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)

                for parent in node._prev:
                    build_topo(parent)

                topo.append(node)

        build_topo(self)

        self.grad = 1.0

        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

In [30]:
a = Value(2.0)
b = Value(3.0)

c = a * b
d = c + a
e = d**2
L = e / b

L.backward()

print("a:", a)
print("b:", b)
print("c:", c)
print("d:", d)
print("e:", e)
print("L:", L)


a: Value(data=2.0, grad=21.333333333333332)
b: Value(data=3.0, grad=3.5555555555555554)
c: Value(data=6.0, grad=5.333333333333333)
d: Value(data=8.0, grad=5.333333333333333)
e: Value(data=64.0, grad=0.3333333333333333)
L: Value(data=21.333333333333332, grad=1.0)


## Gradient Checking

An autograd engine should not be trusted only because its gradients
look reasonable.

We can verify the implementation using independent methods.

### 1. Analytical gradient from our autograd engine

Our `Value` objects construct a computation graph during the forward
pass and apply local derivative rules during the backward pass.

### 2. Numerical gradient with finite differences

For a scalar function $f(x)$, the derivative can be approximated by

$$
\frac{df}{dx}
\approx
\frac{f(x+\epsilon)-f(x-\epsilon)}
{2\epsilon}.
$$

This is called the **central finite-difference approximation**.

For sufficiently small $\epsilon$, it provides an independent numerical
estimate of the derivative.

### 3. PyTorch autograd

We can also express the same computation using PyTorch tensors with

`requires_grad=True`

and compare PyTorch's gradients with our implementation.

If all three methods agree within numerical precision, this provides
strong evidence that the local backward rules and graph traversal are
implemented correctly.

In [31]:
a = Value(2.0)
b = Value(3.0)

L = ((a * b + a) ** 2) / b

L.backward()

print("L:", L.data)
print("dL/da:", a.grad)
print("dL/db:", b.grad)

L: 21.333333333333332
dL/da: 21.333333333333332
dL/db: 3.5555555555555554


In [32]:
def f(a, b):
    return ((a * b + a) ** 2) / b


a0 = 2.0
b0 = 3.0
eps = 1e-6

grad_a_numerical = (f(a0 + eps, b0) - f(a0 - eps, b0)) / (2 * eps)

grad_b_numerical = (f(a0, b0 + eps) - f(a0, b0 - eps)) / (2 * eps)
print("numerical dL/da:", grad_a_numerical)
print("numerical dL/db:", grad_b_numerical)


import torch

a_torch = torch.tensor(
    2.0,
    dtype=torch.float64,
    requires_grad=True,
)

b_torch = torch.tensor(
    3.0,
    dtype=torch.float64,
    requires_grad=True,
)

L_torch = ((a_torch * b_torch + a_torch) ** 2) / b_torch

L_torch.backward()

print("PyTorch L:", L_torch.item())
print("PyTorch dL/da:", a_torch.grad.item())
print("PyTorch dL/db:", b_torch.grad.item())


print("\nOur autograd:")
print(a.grad, b.grad)

print("\nFinite difference:")
print(grad_a_numerical, grad_b_numerical)

print("\nPyTorch:")
print(a_torch.grad.item(), b_torch.grad.item())

numerical dL/da: 21.333333334538906
numerical dL/db: 3.555555554868306
PyTorch L: 21.333333333333332
PyTorch dL/da: 21.333333333333332
PyTorch dL/db: 3.5555555555555554

Our autograd:
21.333333333333332 3.5555555555555554

Finite difference:
21.333333334538906 3.555555554868306

PyTorch:
21.333333333333332 3.5555555555555554


## Building a Neuron with the Autograd Engine

A neuron computes a weighted combination of its inputs followed by a
nonlinear activation function.

For two inputs, we can write

$$
n = w_1x_1 + w_2x_2 + b
$$

and apply a nonlinear activation:

$$
y = \tanh(n).
$$

The parameters are

$$
w_1,\quad w_2,\quad b,
$$

while

$$
x_1,\quad x_2
$$

are the inputs.

During the forward pass, the expression creates a computation graph
from primitive addition and multiplication operations.

During the backward pass, reverse-mode automatic differentiation
computes

$$
\frac{\partial y}{\partial w_1},
\qquad
\frac{\partial y}{\partial w_2},
\qquad
\frac{\partial y}{\partial b}.
$$

These gradients tell us how a small change in each parameter would
change the neuron's output.

The neuron itself does not require a special backward rule. Its
gradient is obtained automatically by composing the backward rules of
multiplication, addition, and `tanh`.

In [33]:
x1: Value = Value(2.0)
x2: Value = Value(0.0)

w1: Value = Value(-3.0)
w2: Value = Value(1.0)
b: Value = Value(6.881373587)

n: Value = x1 * w1 + x2 * w2 + b
y: Value = n.tanh()

print("n:", n)
print("y:", y)
y.backward()

print("x1:", x1)
print("x2:", x2)
print("w1:", w1)
print("w2:", w2)
print("b:", b)
print("n:", n)
print("y:", y)

n: Value(data=0.8813735869999997, grad=0.0)
y: Value(data=0.7071067811767758, grad=0.0)
x1: Value(data=2.0, grad=-1.500000000041458)
x2: Value(data=0.0, grad=0.5000000000138193)
w1: Value(data=-3.0, grad=1.0000000000276386)
w2: Value(data=1.0, grad=0.0)
b: Value(data=6.881373587, grad=0.5000000000138193)
n: Value(data=0.8813735869999997, grad=0.5000000000138193)
y: Value(data=0.7071067811767758, grad=1.0)


## From Primitive Operations to a Neuron

A neuron does not need its own backward rule.

It is a composition of primitive differentiable operations whose local
backward rules are already known.

For inputs

$$
x_1, x_2, \ldots, x_n,
$$

a neuron first computes the affine transformation

$$
z =
\sum_{i=1}^{n} w_i x_i + b.
$$

A nonlinear activation is then applied:

$$
y = \tanh(z).
$$

The trainable parameters are the weights

$$
w_1, w_2, \ldots, w_n
$$

and the bias

$$
b.
$$

Because the computation is constructed from multiplication, addition,
and `tanh`, the existing autograd engine can automatically compute the
gradients of the output with respect to every parameter.

This illustrates the compositional nature of automatic differentiation:
complex differentiable functions can be built from simple primitive
operations without defining a new backward rule for every higher-level
component.

In [37]:
import random
from collections.abc import Sequence


class Neuron:
    def __init__(self, n_inputs: int) -> None:
        # random init weight
        self.weights: list[Value] = [
            Value(random.uniform(-1.0, 1.0)) for _ in range(n_inputs)
        ]
        self.bias: Value = Value(0.0)

    def __call__(self, x: Sequence[Value]) -> Value:
        activation: Value = self.bias

        for weight, value in zip(self.weights, x, strict=True):
            activation = activation + weight * value

        return activation.tanh()

    # trainable parameters
    def parameters(self) -> list[Value]:
        return [*self.weights, self.bias]


neuron: Neuron = Neuron(n_inputs=2)
x: list[Value] = [Value(2.0), Value(-1.0)]

y: Value = neuron(x)

print("output:", y)
print("parameters:", neuron.parameters())

y.backward()

for parameter in neuron.parameters():
    print(parameter)


output: Value(data=-0.9587522973223921, grad=0.0)
parameters: [Value(data=-0.6325349065181427, grad=0.0), Value(data=0.6651639609412399, grad=0.0), Value(data=0.0, grad=0.0)]
Value(data=-0.6325349065181427, grad=0.16158806475807097)
Value(data=0.6651639609412399, grad=-0.08079403237903549)
Value(data=0.0, grad=0.08079403237903549)


## Building a Layer

A neural-network layer contains multiple neurons that receive the same
input vector.

If the input has dimension

$$
n_{\text{in}},
$$

and the layer contains

$$
n_{\text{out}}
$$

neurons, the layer implements a mapping

$$
\mathbb{R}^{n_{\text{in}}}
\rightarrow
\mathbb{R}^{n_{\text{out}}}.
$$

Each neuron has its own weights and bias.

For neuron $j$,

$$
z_j
=
\sum_{i=1}^{n_{\text{in}}}
w_{ji}x_i+b_j,
$$

followed by

$$
y_j=\tanh(z_j).
$$

All neurons receive the same input vector but learn different
parameters.

In [39]:
class Layer:
    def __init__(self, n_inputs: int, n_outputs: int) -> None:
        self.neurons: list[Neuron] = [
            Neuron(n_inputs) for _ in range(n_outputs)
        ]

    def __call__(self, x: Sequence[Value]) -> list[Value]:
        return [neuron(x) for neuron in self.neurons]

    def parameters(self) -> list[Value]:
        return [
            parameter
            for neuron in self.neurons
            for parameter in neuron.parameters()
        ]


layer: Layer = Layer(n_inputs=2, n_outputs=3)

x: list[Value] = [Value(2.0), Value(-1.0)]
outputs: list[Value] = layer(x)

print("outputs:")
for output in outputs:
    print(output)

print(
    "number of parameters:",
    len(layer.parameters()),
)

outputs:
Value(data=0.792925685932999, grad=0.0)
Value(data=0.3403045980032171, grad=0.0)
Value(data=0.7326150804516187, grad=0.0)
number of parameters: 9


## Takeaways

This lesson implemented a minimal scalar reverse-mode automatic
differentiation engine.

The main ideas are:

- The computation graph is created dynamically during the forward pass.
- Each primitive operation stores a local backward rule.
- Local derivatives are combined through the chain rule.
- Gradients must be accumulated because one value may influence the
  final output through multiple paths.
- Reverse-mode autodiff requires traversing the graph in reverse
  topological order.
- Numerical finite differences and PyTorch autograd can be used to
  verify custom gradient implementations.
- Higher-level structures such as neurons and layers do not need their
  own backward rules when they are composed from differentiable
  primitive operations.

The purpose of this implementation is conceptual understanding rather
than computational efficiency.

From this point onward, PyTorch autograd will be used for actual model
implementations.